In [9]:
import os
import json
import urllib.request
import urllib.parse
import streamlit as st
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage, AIMessage
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
import wikipedia

In [2]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
@tool
def get_live_weather(city: str) -> str:
    """Fetch real-time weather data for a specific city name."""
    encoded_city = urllib.parse.quote(city)
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={encoded_city}&count=1&language=en&format=json"
    try:
        with urllib.request.urlopen(geo_url) as geo_response:
            geo_data = json.loads(geo_response.read().decode())
            if not geo_data.get("results"):
                return f"Could not find coordinates for city: {city}"
            location = geo_data["results"][0]
            lat, lon = location["latitude"], location["longitude"]
            resolved_city = location["name"]

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        with urllib.request.urlopen(weather_url) as weather_response:
            weather_data = json.loads(weather_response.read().decode())
            current = weather_data["current_weather"]
            return f"The current temperature in {resolved_city} is {current['temperature']}°C with wind speed {current['windspeed']} km/h."
    except Exception as e:
        return f"Error fetching weather data: {str(e)}"

@tool
def search_arxiv(query: str) -> str:
    """Search the ArXiv academic database for scientific papers across all disciplines.
    
    CRITICAL INSTRUCTION: The query must be STRICTLY keywords or use ArXiv field prefixes. 
    DO NOT use conversational language like 'find papers about'.
    
    Valid query formats:
    - Broad keyword search: 'large language models', 'quantum computing', 'gene editing'
    - Specific fields: 'ti:transformer AND au:vaswani' (Title and Author)
    - Abstract only: 'abs:reinforcement learning'
    - All fields: 'all:computer vision'
    """
    try:
        # Increased top_k_results to retrieve a wider variety of papers
        arxiv = ArxivAPIWrapper(top_k_results=5, doc_content_chars_max=1500)
        result = arxiv.run(query)
        
        # Guide the LLM to self-correct if the search yields nothing
        if not result or "No good Arxiv Result" in result:
            return f"Search failed. The query '{query}' returned no results. Try again using broader keywords or checking your syntax."
            
        return result
    except Exception as e:
        return f"ArXiv API Error: {str(e)}"

wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000)
)

tools = [wikipedia_tool, search_arxiv, get_live_weather]

In [4]:
t_llm=llm.bind_tools(tools)

In [ ]:
tool_node = ToolNode(tools)

def botnode(state: MessagesState):
        response = t_llm.invoke(state["messages"])
        return {"messages": [response]}

def approve_execution(state: MessagesState):
        last_message = state["messages"][-1]
        decision = interrupt({
            "type": "pre_execution",
            "title": "Approval Required: Tool Execution",
            "tool_calls": last_message.tool_calls
        })


In [6]:
rsp=t_llm.invoke("what is attention all you need paper and what it about use the tool?")

In [8]:
rsp.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  search_arxiv (fc_e0adf9b9-12b7-4cca-a9a9-7bbde091f2d1)
 Call ID: fc_e0adf9b9-12b7-4cca-a9a9-7bbde091f2d1
  Args:
    query: Attention Is All You Need
